# RAG 场景 Prefix Caching 模拟器

1. Radix Tree 前缀索引
2. 多轮对话 prefix 复用
3. RAG chunk 缓存命中率
4. 一致性哈希路由模拟

> 阿里 RTP-LLM 核心优化 + RAG serving 系统设计

In [ ]:
import numpy as np
import hashlib
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False
np.random.seed(42)
print("Ready. matplotlib:", HAS_MPL)

## 1) Radix Tree 前缀索引

In [ ]:
@dataclass
class RadixNode:
    tokens: tuple = ()
    kv_block_ids: list = field(default_factory=list)
    children: dict = field(default_factory=dict)
    last_access: int = 0
    ref_count: int = 0

class RadixPrefixCache:
    def __init__(self, max_blocks=512, block_size=16):
        self.root = RadixNode()
        self.max_blocks = max_blocks
        self.block_size = block_size
        self.next_block_id = 0
        self.used_blocks = 0
        self.step = 0
        self.total_queries = 0
        self.total_matched = 0
        self.total_query_tokens = 0
        self.history = []

    def _alloc(self, n_tokens):
        n = int(np.ceil(n_tokens / self.block_size))
        blocks = list(range(self.next_block_id, self.next_block_id + n))
        self.next_block_id += n
        self.used_blocks += n
        return blocks

    def match_prefix(self, tokens):
        node, matched, blocks, remaining = self.root, 0, [], tokens
        while remaining:
            first = remaining[0]
            if first not in node.children:
                break
            child = node.children[first]
            ml = 0
            for i in range(min(len(child.tokens), len(remaining))):
                if remaining[i] == child.tokens[i]: ml += 1
                else: break
            if ml == 0: break
            matched += ml
            blocks.extend(child.kv_block_ids)
            child.last_access = self.step
            child.ref_count += 1
            if ml < len(child.tokens): break
            node, remaining = child, remaining[ml:]
        return matched, blocks

    def insert(self, tokens):
        node, remaining = self.root, tokens
        while remaining:
            first = remaining[0]
            if first not in node.children:
                node.children[first] = RadixNode(tokens=remaining, kv_block_ids=self._alloc(len(remaining)),
                                                  last_access=self.step, ref_count=1)
                return
            child = node.children[first]
            common = 0
            for i in range(min(len(child.tokens), len(remaining))):
                if remaining[i] == child.tokens[i]: common += 1
                else: break
            if common == len(child.tokens):
                remaining = remaining[common:]
                node = child
                continue
            # split
            split = RadixNode(tokens=child.tokens[:common],
                              kv_block_ids=child.kv_block_ids[:int(np.ceil(common/self.block_size))],
                              last_access=self.step, ref_count=child.ref_count)
            child.tokens = child.tokens[common:]
            child.kv_block_ids = child.kv_block_ids[int(np.ceil(common/self.block_size)):]
            split.children[child.tokens[0]] = child
            nr = remaining[common:]
            if nr:
                split.children[nr[0]] = RadixNode(tokens=nr, kv_block_ids=self._alloc(len(nr)),
                                                    last_access=self.step, ref_count=1)
            node.children[split.tokens[0]] = split
            return

    def query(self, tokens):
        self.step += 1
        self.total_queries += 1
        self.total_query_tokens += len(tokens)
        matched, _ = self.match_prefix(tokens)
        self.total_matched += matched
        if matched < len(tokens):
            self.insert(tokens)
        hr = self.total_matched / max(1, self.total_query_tokens)
        self.history.append({"step": self.step, "matched": matched, "qlen": len(tokens),
                             "saved": matched / max(1, len(tokens)), "cum_hr": hr})
        return {"matched": matched, "total": len(tokens), "saved": matched / max(1, len(tokens)),
                "need_prefill": len(tokens) - matched}

# 快速测试
cache = RadixPrefixCache(1024, 16)
sys_p = tuple(range(100))
r1 = cache.query(sys_p + tuple(range(100, 150)))
r2 = cache.query(sys_p + tuple(range(100, 150)) + tuple(range(200, 230)))
print(f"Round 1: matched={r1['matched']}/{r1['total']}, saved={r1['saved']:.0%}")
print(f"Round 2: matched={r2['matched']}/{r2['total']}, saved={r2['saved']:.0%}")

## 2) RAG 工作负载模拟

In [ ]:
def generate_rag_workload(n_queries=500, sys_len=100, n_chunks=50, chunk_len=200,
                          chunks_per_q=3, q_len=(20, 80), zipf_s=1.2, seed=42):
    rng = np.random.default_rng(seed)
    base = 10000
    sys_tok = tuple(range(base, base + sys_len)); base += sys_len
    chunks = []
    for _ in range(n_chunks):
        chunks.append(tuple(range(base, base + chunk_len))); base += chunk_len
    ranks = np.arange(1, n_chunks + 1)
    probs = 1 / np.power(ranks, zipf_s); probs /= probs.sum()
    wl = []
    for i in range(n_queries):
        cidx = sorted(rng.choice(n_chunks, size=chunks_per_q, replace=False, p=probs))
        ql = int(rng.integers(q_len[0], q_len[1] + 1))
        qt = tuple(rng.integers(50000, 60000, size=ql).tolist())
        full = sys_tok
        for c in cidx: full = full + chunks[c]
        full = full + qt
        wl.append({"id": i, "tokens": full, "chunks": list(cidx), "len": len(full)})
    return wl

rag_wl = generate_rag_workload(500)
print(f"Workload: {len(rag_wl)} queries, sample len={rag_wl[0]['len']}, chunks={rag_wl[0]['chunks']}")

## 3) 运行 RAG Prefix Cache 模拟

In [ ]:
cache = RadixPrefixCache(4096, 16)
for req in rag_wl:
    cache.query(req["tokens"])

print(f"Total queries: {cache.total_queries}")
print(f"Token hit rate: {cache.total_matched / cache.total_query_tokens:.1%}")
print(f"Prefill savings: {cache.total_matched:,} / {cache.total_query_tokens:,} tokens")

if HAS_MPL:
    h = cache.history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot([x["step"] for x in h], [x["cum_hr"] for x in h])
    axes[0].set_xlabel("query #"); axes[0].set_ylabel("cumulative hit rate")
    axes[0].set_title("RAG Prefix Cache Hit Rate"); axes[0].grid(alpha=0.3)
    saved = [x["saved"] for x in h]
    axes[1].scatter(range(len(saved)), saved, alpha=0.3, s=5)
    w = 20
    if len(saved) >= w:
        ma = np.convolve(saved, np.ones(w)/w, mode="valid")
        axes[1].plot(range(w-1, len(saved)), ma, "r-", lw=2, label=f"MA({w})")
    axes[1].set_xlabel("query #"); axes[1].set_ylabel("per-query saved ratio")
    axes[1].set_title("Per-query Reuse Ratio"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 4) 一致性哈希路由

In [ ]:
class ConsistentHashRouter:
    def __init__(self, nodes, vnodes=150):
        self.ring = {}
        self.vnodes = vnodes
        for n in nodes:
            for i in range(vnodes):
                h = int(hashlib.md5(f"{n}:{i}".encode()).hexdigest(), 16)
                self.ring[h] = n
        self.sorted_keys = sorted(self.ring.keys())

    def route(self, key):
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        idx = int(np.searchsorted(self.sorted_keys, h))
        if idx >= len(self.sorted_keys): idx = 0
        return self.ring[self.sorted_keys[idx]]

machines = [f"gpu-{i}" for i in range(8)]
router = ConsistentHashRouter(machines)
counts = {m: 0 for m in machines}
for i in range(10000):
    counts[router.route(f"sess_{i}")] += 1
print("Routing distribution:")
for m in machines:
    print(f"  {m}: {counts[m]} ({counts[m]/10000:.1%})")

# 扩容迁移率
old = {f"sess_{i}": router.route(f"sess_{i}") for i in range(10000)}
router2 = ConsistentHashRouter(machines + ["gpu-8"])
changed = sum(1 for i in range(10000) if router2.route(f"sess_{i}") != old[f"sess_{i}"])
print(f"Migration after +1 node: {changed/10000:.1%} (ideal: {1/9:.1%})")

## 5) 多轮对话复用模拟

In [ ]:
def simulate_multi_turn(n_sess=100, max_turns=6, sys_len=100, seed=42):
    rng = np.random.default_rng(seed)
    base = 70000
    sys_tok = tuple(range(base, base + sys_len)); base += sys_len
    reqs = []
    for s in range(n_sess):
        nt = int(rng.integers(2, max_turns + 1))
        history = sys_tok
        for t in range(nt):
            ql = int(rng.integers(15, 60))
            qt = tuple(rng.integers(80000, 90000, size=ql).tolist())
            full = history + qt
            reqs.append({"session": s, "turn": t, "tokens": full, "len": len(full)})
            rl = int(rng.integers(30, 100))
            reply = tuple(rng.integers(90000, 99999, size=rl).tolist())
            history = full + reply
    return reqs

mt_wl = simulate_multi_turn(100, 6)
cache_mt = RadixPrefixCache(8192, 16)
no_cache_total, with_cache_total = 0, 0
for req in mt_wl:
    no_cache_total += req["len"]
    r = cache_mt.query(req["tokens"])
    with_cache_total += r["need_prefill"]
savings = 1 - with_cache_total / no_cache_total
print(f"Multi-turn requests: {len(mt_wl)}")
print(f"Without cache: {no_cache_total:,} prefill tokens")
print(f"With cache:    {with_cache_total:,} prefill tokens")
print(f"Savings:       {savings:.1%}")

## 6) 面试讲解模板

> "RAG/多轮对话场景大量请求共享 system prompt 和文档 chunk。
> 方案：Radix Tree 索引 + 一致性哈希路由 + LRU 驱逐。
> 效果：多轮对话可节省 50-80% prefill，FTT 显著降低。
> 回退：cache miss 退化为完整 prefill，不影响正确性。"

### 追问
1. **Radix Tree vs Hash？** → Hash 只精确匹配；Radix 支持最长前缀
2. **一致性哈希缺点？** → 亲和性 vs 负载均衡冲突
3. **cache 大小？** → 复用率 × 工作集 + 20% buffer
4. **跨机 KV？** → RDMA/GPUDirect 低延迟迁移